# 🌿 PlantHealth AI — Custom CNN from Scratch

## Notebook 01: Building a Baseline Convolutional Neural Network

In this notebook, we build a CNN from scratch to classify plant leaf images into 38 disease categories using the **PlantVillage Dataset**. This serves as our baseline model before attempting Transfer Learning.

### Outline
1. Setup & Data Exploration
2. Data Augmentation Visualization
3. Custom CNN Architecture
4. Training
5. Evaluation & Metrics
6. Confusion Matrix
7. Sample Predictions Grid

---

## 1. Setup & Data Exploration

In [ ]:
import sys
import os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
from PIL import Image

# Project modules
from src.data.dataset import PlantDiseaseDataset, get_data_loaders, get_sample_images
from src.data.transforms import get_train_transforms, get_val_transforms, visualize_augmentations
from src.models.custom_cnn import PlantDiseaseCNN, create_custom_cnn
from src.training.trainer import Trainer, get_optimizer, get_scheduler
from src.evaluation.metrics import evaluate_model, find_misclassified_samples, find_most_confused_classes
from src.utils.visualization import (
    plot_training_history, plot_confusion_matrix, 
    plot_sample_predictions, plot_augmentations, plot_class_distribution
)

# Matplotlib settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load config
config_path = Path(PROJECT_ROOT) / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Paths
TRAIN_DIR = Path(PROJECT_ROOT) / config['paths']['train_dir']
VALID_DIR = Path(PROJECT_ROOT) / config['paths']['valid_dir']
TEST_DIR = Path(PROJECT_ROOT) / config['paths']['test_dir'] if 'test_dir' in config['paths'] else Path(PROJECT_ROOT) / 'data' / 'New Plant Diseases Dataset' / 'test'
CHECKPOINT_DIR = Path(PROJECT_ROOT) / config['paths']['checkpoint_dir']

print(f"Train dir: {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
print(f"Valid dir: {VALID_DIR} (exists: {VALID_DIR.exists()})")
print(f"Test dir:  {TEST_DIR} (exists: {TEST_DIR.exists()})")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
# Load datasets for exploration (with validation transforms for viewing)
val_transforms = get_val_transforms(image_size=config['model']['image_size'])

train_dataset = PlantDiseaseDataset(str(TRAIN_DIR), transform=val_transforms)
val_dataset = PlantDiseaseDataset(str(VALID_DIR), transform=val_transforms, class_to_idx=train_dataset.class_to_idx)

print(f"\nDataset Summary:")
print(f"  Training samples:   {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Number of classes:  {len(train_dataset.classes)}")
print(f"\nClass names:")
for i, name in enumerate(train_dataset.classes):
    print(f"  [{i:2d}] {name}")

In [ ]:
# Plot class distribution
train_dist = train_dataset.get_class_distribution()

fig = plot_class_distribution(
    train_dist, 
    title="Training Set — Class Distribution",
    figsize=(14, 12)
)
plt.show()

print(f"\nMin class size: {min(train_dist.values()):,} images ({min(train_dist, key=train_dist.get)})")
print(f"Max class size: {max(train_dist.values()):,} images ({max(train_dist, key=train_dist.get)})")
print(f"Average class size: {np.mean(list(train_dist.values())):.0f} images")

## 2. Data Augmentation Visualization

Our augmentation pipeline applies random transforms during training to prevent overfitting:
- **Geometric**: Horizontal flip, rotation (±15°), affine transforms
- **Color**: Brightness/contrast jitter, hue/saturation shifts
- **Noise**: Gaussian blur, motion blur, Gaussian noise
- **Advanced**: Optical distortion, grid distortion, elastic transform
- **Cutout**: CoarseDropout to simulate occlusion

In [ ]:
# Get a sample image (raw, no transforms)
samples = get_sample_images(train_dataset, num_samples=1)
original_image, class_name, img_path = samples[0]

print(f"Sample image: {class_name}")
print(f"Image shape: {original_image.shape}")
print(f"Path: {img_path}")

# Get training transforms
train_transforms = get_train_transforms(
    image_size=config['model']['image_size'],
    augmentation_config=config.get('augmentation', {}).get('train', {})
)

# Generate augmented versions
augmented_images = visualize_augmentations(original_image, train_transforms, num_samples=7)

# Plot
fig = plot_augmentations(
    original_image, augmented_images,
    title=f"Data Augmentation — {class_name}",
    figsize=(16, 8)
)
plt.show()

In [ ]:
# Show augmentations for multiple classes
sample_classes = ['Tomato___Early_blight', 'Apple___Black_rot', 'Grape___healthy', 'Corn_(maize)___Common_rust_']

fig, axes = plt.subplots(len(sample_classes), 5, figsize=(18, 4 * len(sample_classes)))

for row, cls_name in enumerate(sample_classes):
    try:
        samples = get_sample_images(train_dataset, num_samples=1, class_name=cls_name)
        orig_img = samples[0][0]
        augs = visualize_augmentations(orig_img, train_transforms, num_samples=4)
        
        # Original
        axes[row, 0].imshow(orig_img if orig_img.max() <= 1 else orig_img / 255.0)
        axes[row, 0].set_title('Original', fontweight='bold', fontsize=10)
        axes[row, 0].set_ylabel(cls_name.replace('___', '\n'), fontsize=9, rotation=0, ha='right')
        axes[row, 0].axis('off')
        
        # Augmented
        for col, aug_img in enumerate(augs):
            axes[row, col + 1].imshow(aug_img)
            axes[row, col + 1].set_title(f'Aug {col+1}', fontsize=10)
            axes[row, col + 1].axis('off')
    except Exception as e:
        print(f"Skipping {cls_name}: {e}")

plt.suptitle('Data Augmentation Across Multiple Classes', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Custom CNN Architecture

Our custom CNN follows this architecture:

```
Input (3, 224, 224)
  → Conv Block 1: Conv2d(3→32) + BatchNorm + ReLU + MaxPool  → (32, 112, 112)
  → Conv Block 2: Conv2d(32→64) + BatchNorm + ReLU + MaxPool → (64, 56, 56)
  → Conv Block 3: Conv2d(64→128) + BatchNorm + ReLU + MaxPool → (128, 28, 28)
  → Conv Block 4: Conv2d(128→256) + BatchNorm + ReLU + MaxPool → (256, 14, 14)
  → AdaptiveAvgPool2d → Flatten → (256,)
  → FC1: Linear(256→512) + BatchNorm + ReLU + Dropout(0.5)
  → FC2: Linear(512→38)
```

In [ ]:
# Create the custom CNN model
num_classes = config['model']['num_classes']
dropout_rate = config['regularization']['dropout']

model = create_custom_cnn(
    num_classes=num_classes,
    dropout_rate=dropout_rate,
    device=device
)

# Print architecture
print("\nFull Architecture:")
print(model)

In [ ]:
# Verify forward pass with a dummy input
dummy_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output (logits): {output[0][:5]}...")  # First 5 logits

## 4. Training

We train the Custom CNN with:
- **Optimizer**: AdamW with learning rate 0.001
- **Scheduler**: ReduceLROnPlateau (patience=3)
- **Callbacks**: EarlyStopping (patience=5), ModelCheckpoint
- **Loss**: CrossEntropyLoss with label smoothing (0.1)
- **Epochs**: 30 (or early stopping)

In [ ]:
# ============================================================
# TRAINING TOGGLE
# Set TRAIN_FROM_SCRATCH = True to train the model from scratch
# Set TRAIN_FROM_SCRATCH = False to load a pre-trained checkpoint
# ============================================================
TRAIN_FROM_SCRATCH = False

CHECKPOINT_PATH = CHECKPOINT_DIR / 'custom_cnn_best.pth'

In [ ]:
# Create data loaders
train_loader, val_loader, class_to_idx, class_names = get_data_loaders(
    train_dir=str(TRAIN_DIR),
    valid_dir=str(VALID_DIR),
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
    image_size=config['model']['image_size'],
    pin_memory=config['training']['pin_memory'],
    augmentation_config=config.get('augmentation', {}).get('train', {}),
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Batch size:    {config['training']['batch_size']}")
print(f"Num classes:   {len(class_names)}")

In [ ]:
if TRAIN_FROM_SCRATCH:
    # Re-create model for training
    model = create_custom_cnn(
        num_classes=num_classes,
        dropout_rate=dropout_rate,
        device=device
    )
    
    # Loss function with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=config['regularization']['label_smoothing'])
    
    # Optimizer
    cnn_config = config['training']['custom_cnn']
    optimizer = get_optimizer(
        model,
        optimizer_name=config['optimizer']['name'],
        learning_rate=cnn_config['learning_rate'],
        weight_decay=cnn_config.get('weight_decay', 0.0001),
    )
    
    # Scheduler
    scheduler = get_scheduler(
        optimizer,
        scheduler_name=config['scheduler']['name'],
        factor=config['scheduler']['factor'],
        patience=config['scheduler']['patience'],
        min_lr=config['scheduler']['min_lr'],
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        checkpoint_dir=str(CHECKPOINT_DIR),
        use_amp=torch.cuda.is_available(),
    )
    
    # Train!
    history = trainer.train(
        epochs=cnn_config['epochs'],
        early_stopping_patience=config['callbacks']['early_stopping']['patience'],
        checkpoint_metric='val_accuracy',
    )
    
    # Save model
    trainer.save_model(str(CHECKPOINT_DIR / 'custom_cnn_best.pth'))
    print("\nTraining complete!")
    
else:
    print(f"Loading pre-trained checkpoint: {CHECKPOINT_PATH}")
    
    if CHECKPOINT_PATH.exists():
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        
        # Handle both formats
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            history = checkpoint.get('history', {})
        else:
            model.load_state_dict(checkpoint)
            history = {}
        
        model.to(device)
        model.eval()
        print("Checkpoint loaded successfully!")
        
        if history:
            print(f"Training epochs recorded: {len(history.get('train_loss', []))}")
            if 'val_acc' in history:
                print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")
    else:
        print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")
        print("Set TRAIN_FROM_SCRATCH = True to train the model")
        history = {}

In [ ]:
# Plot training history (if available)
if history and 'train_loss' in history and len(history['train_loss']) > 0:
    fig = plot_training_history(
        history,
        title='Custom CNN — Training History',
        save_path=str(Path(PROJECT_ROOT) / 'reports' / 'figures' / 'custom_cnn_training_curves.png')
    )
    plt.show()
else:
    print("No training history available. Train from scratch to see training curves.")

## 5. Evaluation & Metrics

We evaluate the trained model on the **test set** to get unbiased performance metrics.

In [ ]:
# Create test data loader
from src.data.dataset import PlantDiseaseDataset
from torch.utils.data import DataLoader

test_dataset = PlantDiseaseDataset(
    str(TEST_DIR),
    transform=val_transforms,
    class_to_idx=class_to_idx,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['training']['num_workers'],
    pin_memory=True,
)

print(f"Test samples: {len(test_dataset):,}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Run comprehensive evaluation
model.eval()
eval_results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=class_names,
    print_report=True,
)

In [ ]:
# Summary metrics
print("\n" + "="*50)
print("CUSTOM CNN — SUMMARY METRICS")
print("="*50)
print(f"Overall Accuracy:     {eval_results['accuracy']:.2f}%")
print(f"Top-5 Accuracy:       {eval_results['top5_accuracy']:.2f}%")
print(f"Macro Precision:      {eval_results['macro_precision']:.4f}")
print(f"Macro Recall:         {eval_results['macro_recall']:.4f}")
print(f"Macro F1-Score:       {eval_results['macro_f1']:.4f}")
print(f"Weighted F1-Score:    {eval_results['weighted_f1']:.4f}")

## 6. Confusion Matrix

In [ ]:
# Ensure reports directory exists
reports_dir = Path(PROJECT_ROOT) / 'reports' / 'figures'
reports_dir.mkdir(parents=True, exist_ok=True)

# Plot confusion matrix
fig = plot_confusion_matrix(
    eval_results['confusion_matrix'],
    class_names=class_names,
    title='Custom CNN — Confusion Matrix (Normalized)',
    normalize=True,
    save_path=str(reports_dir / 'custom_cnn_confusion_matrix.png'),
)
plt.show()

In [ ]:
# Most confused class pairs
confused_pairs = find_most_confused_classes(
    eval_results['confusion_matrix'],
    class_names,
    top_n=10,
)

print("\nTop 10 Most Confused Class Pairs:")
print("-" * 70)
for i, pair in enumerate(confused_pairs, 1):
    print(f"{i:2d}. True: {pair['true_class'][:35]:35s}  →  Pred: {pair['predicted_class'][:35]:35s}  ({pair['count']} times)")

## 7. Sample Predictions Grid

Visualizing 10 random test images with their true labels, predicted labels, and confidence.

In [ ]:
from src.data.transforms import denormalize

# Get 10 random test samples
np.random.seed(42)
indices = np.random.choice(len(test_dataset), size=10, replace=False)

images_list = []
true_labels_list = []
pred_labels_list = []
confidence_list = []

model.eval()
with torch.no_grad():
    for idx in indices:
        image, label = test_dataset[idx]
        
        # Get prediction
        output = model(image.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs.max().item() * 100
        
        # Denormalize image for display
        img_display = denormalize(image.numpy())
        
        images_list.append(img_display)
        true_labels_list.append(class_names[label])
        pred_labels_list.append(class_names[pred_idx])
        confidence_list.append(confidence)

# Plot
fig = plot_sample_predictions(
    images=images_list,
    true_labels=true_labels_list,
    pred_labels=pred_labels_list,
    confidences=confidence_list,
    title='Custom CNN — Sample Test Predictions',
    figsize=(18, 8),
    save_path=str(reports_dir / 'custom_cnn_sample_predictions.png'),
)
plt.show()

# Print results
correct = sum(1 for t, p in zip(true_labels_list, pred_labels_list) if t == p)
print(f"\nCorrect: {correct}/10")

In [ ]:
# Save key metrics for comparison with transfer learning
import json

custom_cnn_results = {
    'model': 'Custom CNN (PlantDiseaseCNN)',
    'accuracy': eval_results['accuracy'],
    'top5_accuracy': eval_results['top5_accuracy'],
    'macro_precision': float(eval_results['macro_precision']),
    'macro_recall': float(eval_results['macro_recall']),
    'macro_f1': float(eval_results['macro_f1']),
    'weighted_f1': float(eval_results['weighted_f1']),
}

results_path = Path(PROJECT_ROOT) / 'reports' / 'custom_cnn_results.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(custom_cnn_results, f, indent=2)

print(f"Results saved to: {results_path}")
print(json.dumps(custom_cnn_results, indent=2))

---

## Summary

In this notebook, we:
1. ✅ Explored the PlantVillage dataset (38 classes, 50k+ images)
2. ✅ Visualized the data augmentation pipeline
3. ✅ Built a custom 4-layer CNN from scratch
4. ✅ Trained (or loaded) the model with regularization
5. ✅ Evaluated with comprehensive metrics
6. ✅ Generated confusion matrix and sample predictions

**Next step**: `02_Transfer_Learning.ipynb` — Using ResNet50 for higher accuracy (>90%).